# 10 - Check faster serving

Keep the saved model. The API now reuses a thread controller and transforms each batch once. Check correctness, build the updated image, and measure the same local workload. The target is still p95 below 300 ms at concurrency 10. No training or Azure resources.


## 1. Check correctness

Run the correctness tests in a fresh process. No kernel restart is needed. Stop on failures or skips. The result is saved in `reports/serving-check-tests.txt`.


In [1]:
from pathlib import Path
import json
import platform
import subprocess
import sys

assert platform.system() == "Linux", "Use the WSL kernel."
assert Path(sys.prefix).name == "mlop-pj", "Select the project environment."
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents]
            if p.name == "MLOP-PJ" and (p / "README.md").is_file())
import os
result = subprocess.run([sys.executable, "-m", "unittest", "discover", "-s", "tests", "-v"],
                        cwd=ROOT, capture_output=True, text=True,
                        env={**os.environ, "OMP_NUM_THREADS": "1", "OPENBLAS_NUM_THREADS": "1"})
(ROOT / "reports/serving-check-tests.txt").write_text(result.stdout + result.stderr)
print((result.stdout + result.stderr)[-1000:])
assert result.returncode == 0 and "skipped" not in result.stderr.lower(), "Review failed or skipped tests."


est_duplicate_groups_and_split_are_repeatable (test_workflow.DataTests.test_duplicate_groups_and_split_are_repeatable) ... ok
test_numeric_errors_are_not_repaired (test_workflow.DataTests.test_numeric_errors_are_not_repaired) ... ok
test_preparation_checks_hash_and_excludes_ids (test_workflow.DataTests.test_preparation_checks_hash_and_excludes_ids) ... ok
test_quarantine_preserves_bad_row_and_zeros (test_workflow.DataTests.test_quarantine_preserves_bad_row_and_zeros) ... ok
test_imputer_and_scaler_fit_only_training_data (test_workflow.ModelTests.test_imputer_and_scaler_fit_only_training_data) ... ok
test_predict_rejects_bad_contract_and_accepts_named_order (test_workflow.ModelTests.test_predict_rejects_bad_contract_and_accepts_named_order) ... ok
test_study_ignores_test_data_and_bundle_reload_matches (test_workflow.ModelTests.test_study_ignores_test_data_and_bundle_reload_matches) ... ok

----------------------------------------------------------------------
Ran 15 tests in 1.385s

OK


## 2. Build and measure

The source changed, so build a new image. Docker can reuse the installed package layer. Keep 1 CPU, 512 MB, 200 requests, and concurrency 10. The script checks golden predictions and batch 100 before measuring. Outputs go to `reports/serving-recheck/`; earlier results stay unchanged. The container stops when done. Stop if this cell fails.


In [2]:
result = subprocess.run([sys.executable, "scripts/docker_check.py",
                         "--report-dir", "reports/serving-recheck",
                         "--requests", "200", "--concurrency", "10"],
                        cwd=ROOT, capture_output=True, text=True)
print(result.stdout)
if result.stderr:
    print(result.stderr)
assert result.returncode == 0, "Review serving-recheck/docker-build.txt or docker-api.log."


Pulling python:3.13-slim and building the API image. This may take a few minutes.
HTTP smoke passed. Measuring 200 requests at concurrency 10 by default.
{
  "scope": "local real HTTP; not Azure performance",
  "model_version": "abfd88e893ca",
  "requests": 200,
  "concurrency": 10,
  "batch_size": 1,
  "warmup_requests": 10,
  "errors": 0,
  "error_rate": 0.0,
  "duration_seconds": 2.650111864997598,
  "successful_requests_per_second": 75.46851234530105,
  "mean_ms": 130.3323092498249,
  "p95_ms": 274.54152094978775,
  "latency_scope": "successful requests; client HTTP time including response parsing",
  "client_setup": "shared verified TLS context; per-thread opener reused; setup outside request timer",
  "workload": "cycle five saved golden profiles; closed-loop clients",
  "target_p95_ms": 300,
  "target_met": true,
  "image_id": "sha256:689d113ccc3d3b024682d2c6ea632c00d4bc0a07c3f89c86d1596d23b9872d36",
  "created_at": "2026-10-11T12:48:55.542667+00:00",
  "checker_sha256": "73f15d

## 3. Compare results

Expect the same model artifact, smoke passed, and zero errors. The image changes because the serving source changed. Read p95 and `target_met`; do not assume the optimization passes. Save this notebook. You can also run the read-only Azure checks in 11.


In [3]:
old_image = json.loads((ROOT / "reports/history/09-initial/docker-image.json").read_text())
old_load = json.loads((ROOT / "reports/history/09-initial/local-load.json").read_text())
folder = ROOT / "reports/serving-recheck"
image = json.loads((folder / "docker-image.json").read_text())
smoke = json.loads((folder / "docker-smoke.json").read_text())
load = json.loads((folder / "local-load.json").read_text())
assert image["artifact_sha256"] == old_image["artifact_sha256"]
assert image["model_version"] == old_image["model_version"] == load["model_version"]
assert image["image_id"] == smoke["image_id"] == load["image_id"]
assert smoke["passed"] and load["errors"] == 0
assert (load["requests"], load["concurrency"], load["target_p95_ms"]) == (200, 10, 300)
print(json.dumps({"model_unchanged": True, "smoke_passed": smoke["passed"],
                  "before_p95_ms": old_load["p95_ms"], "after_p95_ms": load["p95_ms"],
                  "errors": load["errors"], "target_met": load["target_met"]}, indent=2))


{
  "model_unchanged": true,
  "smoke_passed": true,
  "before_p95_ms": 496.01491619941953,
  "after_p95_ms": 274.54152094978775,
  "errors": 0,
  "target_met": true
}
